Goal: Load arbitrary .msh file

In [1]:
using Pkg
using Revise

In [1]:
using FerriteGmsh

Info    : Reading 'circle.msh'...
Info    : 3 entities
Info    : 1550 nodes
Info    : 3098 elements
Info    : Done reading 'circle.msh'


Ferrite.Grid{2, Ferrite.Triangle, Float64} with 2972 Ferrite.Triangle cells and 1550 nodes

In [10]:
grid_circ = togrid("circle.msh")
∂Ω_circ = union(getfacetset.((grid,), ["boundary"])...)
fe_circ = FerriteFESpace{RefTriangle}(grid_circ, 2, 3, ∂Ω_circ)

Info    : Reading 'circle.msh'...
Info    : 3 entities
Info    : 1550 nodes
Info    : 3098 elements
Info    : Done reading 'circle.msh'


FerriteFESpace{RefTriangle}(CellValues{Ferrite.FunctionValues{1, Lagrange{RefTriangle, 2}, Matrix{Float64}, Matrix{Vec{2, Float64}}, Matrix{Vec{2, Float64}}, Nothing, Nothing}, Ferrite.GeometryMapping{1, Lagrange{RefTriangle, 1}, Matrix{Float64}, Matrix{Vec{2, Float64}}, Nothing}, QuadratureRule{RefTriangle, Vector{Float64}, Vector{Vec{2, Float64}}}, Vector{Float64}}(Ferrite.FunctionValues{1, Lagrange{RefTriangle, 2}, Matrix{Float64}, Matrix{Vec{2, Float64}}, Matrix{Vec{2, Float64}}, Nothing, Nothing}(Lagrange{RefTriangle, 2}(), [-0.11111111111111223 -0.12 -0.12 0.11999999999999997; -0.11111111111111223 -0.12 0.11999999999999997 -0.12; … ; 0.44444444444444897 0.4800000000000001 0.48000000000000015 0.16000000000000003; 0.44444444444444897 0.4800000000000001 0.16000000000000006 0.48], [-0.11111111111111223 -0.12 -0.12 0.11999999999999997; -0.11111111111111223 -0.12 0.11999999999999997 -0.12; … ; 0.44444444444444897 0.4800000000000001 0.48000000000000015 0.16000000000000003; 0.44444444444

In [ ]:
using ModularEIT
using FFTW
using Images
using Ferrite
using Enzyme
using LinearAlgebra

In [ ]:
n = 63
#grid = generate_grid(Triangle, (n, n))
grid = generate_grid(Quadrilateral, (n, n))
∂Ω = union(getfacetset.((grid,), ["left", "top", "right", "bottom"])...)

println("Type of ∂Ω: $(typeof(∂Ω))")
#fe = FerriteFESpace{RefTriangle}(grid, 2, 3, ∂Ω)
fe = FerriteFESpace{RefQuadrilateral}(grid, 2, 3, ∂Ω)
img = load("../../test/SolverTests/Reference2Spot.jpg")
itp = interpolate_array_2D(Float64.(img))
cond_vec = project_function_to_fem(fe, itp)

KeyError: KeyError: key "left" not found

In [4]:
G_full = real_fourier_basis(8)

256×256 Matrix{Float64}:
 0.0625  0.0883883   0.0         …   0.0883883   0.0          0.0625
 0.0625  0.0883617   0.00216916     -0.0883617   0.00216916  -0.0625
 0.0625  0.0882819   0.00433701      0.0882819  -0.00433701   0.0625
 0.0625  0.0881489   0.00650225     -0.0881489   0.00650225  -0.0625
 0.0625  0.0879627   0.00866357      0.0879627  -0.00866357   0.0625
 0.0625  0.0877236   0.0108197   …  -0.0877236   0.0108197   -0.0625
 0.0625  0.0874317   0.0129693       0.0874317  -0.0129693    0.0625
 0.0625  0.0870871   0.015111       -0.0870871   0.015111    -0.0625
 0.0625  0.08669     0.0172437       0.08669    -0.0172437    0.0625
 0.0625  0.0862407   0.019366       -0.0862407   0.019366    -0.0625
 ⋮                               ⋱                            ⋮
 0.0625  0.0862407  -0.019366       -0.0862407  -0.019366    -0.0625
 0.0625  0.08669    -0.0172437       0.08669     0.0172437    0.0625
 0.0625  0.0870871  -0.015111       -0.0870871  -0.015111    -0.0625
 0.0625  0.087

In [6]:
G_full = real_fourier_basis(8)
rhs_vec = Vector{Any}(undef, 255)

Threads.@threads for i in 2:256
    M = make_boundary(G_full[:, i], 64)
    itp = interpolate_array_2D(M)
    rhs_vec[i-1] = assemble_rhs_func(fe, itp)
end
# Assemble stiffness matrix and calculate boundary pairs:
K = assemble_L(fe, cond_vec)
K_fac = factorize(K)

mode_vec = Vector{Any}(undef, 255)
for i in 1:255
    mode_vec[i] = create_mode_from_g(fe, rhs_vec[i], K)
end
# define starting guess and define problem:
σ_vec = project_function_to_fem(fe, x -> 0.5)
sol = FerriteSolverState(fe, σ_vec)
prblm = FerriteProblem(fe, mode_vec, sol)


FerriteProblem(FerriteFESpace{RefQuadrilateral}(CellValues{Ferrite.FunctionValues{1, Lagrange{RefQuadrilateral, 2}, Matrix{Float64}, Matrix{Vec{2, Float64}}, Matrix{Vec{2, Float64}}, Nothing, Nothing}, Ferrite.GeometryMapping{1, Lagrange{RefQuadrilateral, 1}, Matrix{Float64}, Matrix{Vec{2, Float64}}, Nothing}, QuadratureRule{RefQuadrilateral, Vector{Float64}, Vector{Vec{2, Float64}}}, Vector{Float64}}(Ferrite.FunctionValues{1, Lagrange{RefQuadrilateral, 2}, Matrix{Float64}, Matrix{Vec{2, Float64}}, Matrix{Vec{2, Float64}}, Nothing, Nothing}(Lagrange{RefQuadrilateral, 2}(), [0.47237900077244527 -1.4605338811812958e-17 … 1.8551212633834227e-18 0.007620999227554982; -0.05999999999999999 1.4605338811812958e-17 … -1.8551212633834227e-18 -0.059999999999999984; … ; 0.2749193338482966 -8.50014503228635e-18 … -8.500145032286352e-18 -0.03491933384829665; 0.15999999999999984 0.3999999999999998 … 0.3999999999999999 0.15999999999999992], [0.47237900077244527 -1.4605338811812958e-17 … 1.855121263383

In [7]:
    eval_points = reshape(equidistant_grid(64), :)
    ph = PointEvalHandler(grid, eval_points)

PointEvalHandler{Grid{2, Quadrilateral, Float64}, Float64}
  number of points: 4096
  Found corresponding cell for all points.

In [15]:
prblm.state.σ

16129-element Vector{Float64}:
 0.5000000000000001
 0.5000000000000002
 0.5000000000000001
 0.49999999999999994
 0.5
 0.5000000000000001
 0.5000000000000001
 0.5000000000000002
 0.5
 0.5
 ⋮
 0.5
 0.49999999999999956
 0.5000000000000003
 0.5000000000000002
 0.4999999999999998
 0.4999999999999998
 0.5000000000000003
 0.5000000000000003
 0.4999999999999999

In [16]:
f, ∂f = create_f∂f(prblm, 100; regularize=false, gn=true) 

(ModularEIT.var"#46#47"{Bool, String, typeof(objective_neumann_init!), FerriteProblem, Int64, Base.RefValue{Bool}}(false, "neumann", ModularEIT.objective_neumann_init!, FerriteProblem(FerriteFESpace{RefQuadrilateral}(CellValues{Ferrite.FunctionValues{1, Lagrange{RefQuadrilateral, 2}, Matrix{Float64}, Matrix{Vec{2, Float64}}, Matrix{Vec{2, Float64}}, Nothing, Nothing}, Ferrite.GeometryMapping{1, Lagrange{RefQuadrilateral, 1}, Matrix{Float64}, Matrix{Vec{2, Float64}}, Nothing}, QuadratureRule{RefQuadrilateral, Vector{Float64}, Vector{Vec{2, Float64}}}, Vector{Float64}}(Ferrite.FunctionValues{1, Lagrange{RefQuadrilateral, 2}, Matrix{Float64}, Matrix{Vec{2, Float64}}, Matrix{Vec{2, Float64}}, Nothing, Nothing}(Lagrange{RefQuadrilateral, 2}(), [0.47237900077244527 -1.4605338811812958e-17 … 1.8551212633834227e-18 0.007620999227554982; -0.05999999999999999 1.4605338811812958e-17 … -1.8551212633834227e-18 -0.059999999999999984; … ; 0.2749193338482966 -8.50014503228635e-18 … -8.500145032286352e

In [18]:
f(σ_vec)

0.0942716856642109

In [19]:
∂f(σ_vec)

16129-element Vector{Float64}:
  1.0858221025158188e-5
  8.007834096017364e-6
  8.458129652577019e-6
  1.5604416359361154e-5
  2.720124231363711e-6
  8.922408939981835e-6
  1.2194585653681183e-5
  6.527476258220192e-6
  9.19836304282792e-6
  3.976927535849789e-6
  ⋮
 -6.396255951929813e-6
 -2.9633159945625064e-6
 -5.185224715112531e-7
 -6.106382505398392e-6
 -2.6454255143119505e-6
 -1.3261073563015577e-5
 -2.756135531062671e-6
 -4.330901017586967e-6
 -1.1205633760568072e-7